# Statistical Analysis

In [ ]:
import pandas as pd
import numpy as np
import ast
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import seaborn as sns


In [ ]:
BG_COLOR = "#23272e"  
BOX_COLOR = "#61afef"  
TEXT_COLOR = "#abb2bf"  

plt.rcParams.update(
    {
        "figure.facecolor": BG_COLOR,  
        "axes.facecolor": BG_COLOR, 
        "axes.edgecolor": "#3f444a",  
        "axes.labelcolor": TEXT_COLOR,  
        "text.color": TEXT_COLOR,  
        "xtick.color": TEXT_COLOR,
        "ytick.color": TEXT_COLOR,  
        "grid.color": "#2c313c",  
    }
)

sns.set_theme(
    style="dark",
    rc={
        "figure.facecolor": BG_COLOR,
        "axes.facecolor": BG_COLOR,
        "text.color": "#abb2bf",
        "axes.labelcolor": "#abb2bf",
        "xtick.color": "#abb2bf",
        "ytick.color": "#abb2bf",
    },
)

In [ ]:
initial_df = pd.read_parquet("../data/processed/stats_df.parquet")
initial_df

df = initial_df
df.info()

## Anime Adaptation Relations (Linear Regression)

In [ ]:
adaptation_df = df[~df['adaptation_score'].isna()]
adaptation_df.plot.scatter(x='adaptation_score', y='score', title='Source Material Score vs. Anime Score')
plt.show()

We will now perform a Linear Regression:

In [ ]:
X = adaptation_df[['adaptation_score']]
y = adaptation_df[['score']]

model = LinearRegression()
model.fit(X, y)

r2 = model.score(X, y)  
print(f"Correct R^2 Score: {r2}")

plt.scatter(X, y)
plt.plot(X, model.predict(X), color='red', linewidth=2)
plt.title('Source Material Score vs. Anime Score')
plt.show()

We perform the same analysis on member count vs WC score:

In [ ]:
adaptation_df = df[~df['adaptation_members'].isna()]
adaptation_df.plot.scatter(x='adaptation_members', y='wc', title='Source Material Members vs. WC Score')
plt.show()

In [ ]:
X = adaptation_df[['adaptation_members']]
y = adaptation_df[['wc']]

model = LinearRegression()
model.fit(X, y)

r2 = model.score(X, y)  
print(f"Correct R^2 Score: {r2}")

plt.scatter(X, y)
plt.plot(X, model.predict(X), color='red', linewidth=2)
plt.title('Source Material Members vs. Anime WC')
plt.show()

In [ ]:
prequel_df = df[~df['prequel_score'].isna()]
prequel_df.plot.scatter(x='prequel_score', y='score', title='Prequel Score vs. Anime Score')
plt.show()

In [ ]:
X = prequel_df[['prequel_score']]
y = prequel_df[['score']]

model = LinearRegression()
model.fit(X, y)

r2 = model.score(X, y)  
print(f"Correct R^2 Score: {r2}")

plt.scatter(X, y)
plt.plot(X, model.predict(X), color='red', linewidth=2)
plt.title('Prequel Score vs. Anime Score')
plt.show()

In [ ]:
prequel_df = df[~df['prequel_wc'].isna()]
prequel_df.plot.scatter(x='prequel_wc', y='wc', title='Prequel WC vs. Anime WC (Log-log)')
plt.show()

In [ ]:
X = prequel_df[['prequel_wc']]
y = prequel_df[['wc']]

model = LinearRegression()
model.fit(X, y)

r2 = model.score(X, y)  
print(f"Correct R^2 Score: {r2}")

plt.scatter(X, y)
plt.plot(X, model.predict(X), color='red', linewidth=2)
plt.title('Prequel WC vs. Anime WC (Log-log)')
plt.show()

## Genre, Theme, Demographic vs. Score (Multiple Regression)

In [ ]:
genre_averages = df.explode('genres').groupby('genres')['score'].mean().sort_values()
genre_counts = df['genres'].explode().value_counts()
print(genre_counts)
genre_averages

In [ ]:
distinct_genres = sorted({
    genre
    for genres in df['genres']
    for genre in genres
})

genre_matrix = pd.DataFrame(
    0,
    index=df.index,
    columns=distinct_genres,
    dtype=int
)

for idx, genres in df['genres'].items():
    if len(genres) > 0:  # handles [] naturally
        genre_matrix.loc[idx, genres] = 1


min_count = 50

genre_counts = genre_matrix.sum(axis=0)

valid_genres = genre_counts[
    genre_counts >= min_count
].index.tolist()

dropped = set(distinct_genres) - set(valid_genres)

if dropped:
    print(
        f"Dropping {len(dropped)} rare genres from regression: "
        f"{sorted(dropped)}"
    )

X = genre_matrix[valid_genres].copy()

X = sm.add_constant(X)

y = df['score']

model = sm.OLS(y, X, missing='drop').fit()

print(model.summary())

results = pd.DataFrame({
    'genre': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("genre != 'const'")


if not results.empty:
    results['pval_fdr'] = multipletests(
        results['pval'],
        method='fdr_bh'
    )[1]

    results['significant'] = results['pval_fdr'] < 0.05

else:
    results['pval_fdr'] = pd.Series(dtype=float)
    results['significant'] = pd.Series(dtype=bool)


results = results.sort_values('coef', ascending=False)

results

In [ ]:
distinct_themes = sorted({
    theme
    for themes in df['themes']
    for theme in themes
})

theme_matrix = pd.DataFrame(
    0,
    index=df.index,
    columns=distinct_themes,
    dtype=int
)

for idx, themes in df['themes'].items():
    if len(themes) > 0:  # handles [] naturally
        theme_matrix.loc[idx, themes] = 1


min_count = 50

theme_counts = theme_matrix.sum(axis=0)

valid_themes = theme_counts[
    theme_counts >= min_count
].index.tolist()

dropped = set(distinct_themes) - set(valid_themes)

if dropped:
    print(
        f"Dropping {len(dropped)} rare themes from regression: "
        f"{sorted(dropped)}"
    )

X = theme_matrix[valid_themes].copy()

X = sm.add_constant(X)

y = df['score']

model = sm.OLS(y, X, missing='drop').fit()

print(model.summary())

results = pd.DataFrame({
    'theme': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("theme != 'const'")


if not results.empty:
    results['pval_fdr'] = multipletests(
        results['pval'],
        method='fdr_bh'
    )[1]

    results['significant'] = results['pval_fdr'] < 0.05

else:
    results['pval_fdr'] = pd.Series(dtype=float)
    results['significant'] = pd.Series(dtype=bool)


results = results.sort_values('coef', ascending=False)

results

In [ ]:
distinct_demos = sorted({
    demo
    for demos in df['demographics']
    for demo in demos
})

demo_matrix = pd.DataFrame(
    0,
    index=df.index,
    columns=distinct_demos,
    dtype=int
)

for idx, demos in df['demographics'].items():
    if len(demos) > 0:  # handles [] naturally
        demo_matrix.loc[idx, demos] = 1

X = demo_matrix.copy()

X = sm.add_constant(X)

y = df['score']

model = sm.OLS(y, X, missing='drop').fit()

print(model.summary())

results = pd.DataFrame({
    'demographic': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("demographic != 'const'")


if not results.empty:
    results['pval_fdr'] = multipletests(
        results['pval'],
        method='fdr_bh'
    )[1]

    results['significant'] = results['pval_fdr'] < 0.05

else:
    results['pval_fdr'] = pd.Series(dtype=float)
    results['significant'] = pd.Series(dtype=bool)


results = results.sort_values('coef', ascending=False)

results

## Genre, Theme, Demographic vs. WC (Multiple Regression)

In [ ]:
distinct_genres = sorted({
    genre
    for genres in df['genres']
    for genre in genres
})

genre_matrix = pd.DataFrame(
    0,
    index=df.index,
    columns=distinct_genres,
    dtype=int
)

for idx, genres in df['genres'].items():
    if len(genres) > 0:  # handles [] naturally
        genre_matrix.loc[idx, genres] = 1


min_count = 50

genre_counts = genre_matrix.sum(axis=0)

valid_genres = genre_counts[
    genre_counts >= min_count
].index.tolist()

dropped = set(distinct_genres) - set(valid_genres)

if dropped:
    print(
        f"Dropping {len(dropped)} rare genres from regression: "
        f"{sorted(dropped)}"
    )

X = genre_matrix[valid_genres].copy()

X = sm.add_constant(X)

y = df['wc']

model = sm.OLS(y, X, missing='drop').fit()

print(model.summary())

results = pd.DataFrame({
    'genre': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("genre != 'const'")


if not results.empty:
    results['pval_fdr'] = multipletests(
        results['pval'],
        method='fdr_bh'
    )[1]

    results['significant'] = results['pval_fdr'] < 0.05

else:
    results['pval_fdr'] = pd.Series(dtype=float)
    results['significant'] = pd.Series(dtype=bool)


results = results.sort_values('coef', ascending=False)

results

In [ ]:
distinct_themes = sorted({
    theme
    for themes in df['themes']
    for theme in themes
})

theme_matrix = pd.DataFrame(
    0,
    index=df.index,
    columns=distinct_themes,
    dtype=int
)

for idx, themes in df['themes'].items():
    if len(themes) > 0:  # handles [] naturally
        theme_matrix.loc[idx, themes] = 1


min_count = 50

theme_counts = theme_matrix.sum(axis=0)

valid_themes = theme_counts[
    theme_counts >= min_count
].index.tolist()

dropped = set(distinct_themes) - set(valid_themes)

if dropped:
    print(
        f"Dropping {len(dropped)} rare themes from regression: "
        f"{sorted(dropped)}"
    )

X = theme_matrix[valid_themes].copy()

X = sm.add_constant(X)

y = df['wc']

model = sm.OLS(y, X, missing='drop').fit()

print(model.summary())

results = pd.DataFrame({
    'theme': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("theme != 'const'")


if not results.empty:
    results['pval_fdr'] = multipletests(
        results['pval'],
        method='fdr_bh'
    )[1]

    results['significant'] = results['pval_fdr'] < 0.05

else:
    results['pval_fdr'] = pd.Series(dtype=float)
    results['significant'] = pd.Series(dtype=bool)


results = results.sort_values('coef', ascending=False)

results

In [ ]:
distinct_demos = sorted({
    demo
    for demos in df['demographics']
    for demo in demos
})

demo_matrix = pd.DataFrame(
    0,
    index=df.index,
    columns=distinct_demos,
    dtype=int
)

for idx, demos in df['demographics'].items():
    if len(demos) > 0:  # handles [] naturally
        demo_matrix.loc[idx, demos] = 1

X = demo_matrix.copy()

X = sm.add_constant(X)

y = df['wc']

model = sm.OLS(y, X, missing='drop').fit()

print(model.summary())

results = pd.DataFrame({
    'demographic': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("demographic != 'const'")


if not results.empty:
    results['pval_fdr'] = multipletests(
        results['pval'],
        method='fdr_bh'
    )[1]

    results['significant'] = results['pval_fdr'] < 0.05

else:
    results['pval_fdr'] = pd.Series(dtype=float)
    results['significant'] = pd.Series(dtype=bool)


results = results.sort_values('coef', ascending=False)

results

## Genre, Theme, and Demographic vs. Drop Rate (Multiple Regression)

In [ ]:
distinct_genres = sorted({
    genre
    for genres in df['genres']
    for genre in genres
})

genre_matrix = pd.DataFrame(
    0,
    index=df.index,
    columns=distinct_genres,
    dtype=int
)

for idx, genres in df['genres'].items():
    if len(genres) > 0:  # handles [] naturally
        genre_matrix.loc[idx, genres] = 1


min_count = 50

genre_counts = genre_matrix.sum(axis=0)

valid_genres = genre_counts[
    genre_counts >= min_count
].index.tolist()

dropped = set(distinct_genres) - set(valid_genres)

if dropped:
    print(
        f"Dropping {len(dropped)} rare genres from regression: "
        f"{sorted(dropped)}"
    )

X = genre_matrix[valid_genres].copy()

X = sm.add_constant(X)

y = df['drop_rate']

model = sm.OLS(y, X, missing='drop').fit()

print(model.summary())

results = pd.DataFrame({
    'genre': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("genre != 'const'")


if not results.empty:
    results['pval_fdr'] = multipletests(
        results['pval'],
        method='fdr_bh'
    )[1]

    results['significant'] = results['pval_fdr'] < 0.05

else:
    results['pval_fdr'] = pd.Series(dtype=float)
    results['significant'] = pd.Series(dtype=bool)


results = results.sort_values('coef', ascending=False)

results

In [ ]:
distinct_themes = sorted({
    theme
    for themes in df['themes']
    for theme in themes
})

theme_matrix = pd.DataFrame(
    0,
    index=df.index,
    columns=distinct_themes,
    dtype=int
)

for idx, themes in df['themes'].items():
    if len(themes) > 0:  # handles [] naturally
        theme_matrix.loc[idx, themes] = 1


min_count = 50

theme_counts = theme_matrix.sum(axis=0)

valid_themes = theme_counts[
    theme_counts >= min_count
].index.tolist()

dropped = set(distinct_themes) - set(valid_themes)

if dropped:
    print(
        f"Dropping {len(dropped)} rare themes from regression: "
        f"{sorted(dropped)}"
    )

X = theme_matrix[valid_themes].copy()

X = sm.add_constant(X)

y = df['drop_rate']

model = sm.OLS(y, X, missing='drop').fit()

print(model.summary())

results = pd.DataFrame({
    'theme': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("theme != 'const'")


if not results.empty:
    results['pval_fdr'] = multipletests(
        results['pval'],
        method='fdr_bh'
    )[1]

    results['significant'] = results['pval_fdr'] < 0.05

else:
    results['pval_fdr'] = pd.Series(dtype=float)
    results['significant'] = pd.Series(dtype=bool)


results = results.sort_values('coef', ascending=False)

results

In [ ]:
distinct_demos = sorted({
    demo
    for demos in df['demographics']
    for demo in demos
})

demo_matrix = pd.DataFrame(
    0,
    index=df.index,
    columns=distinct_demos,
    dtype=int
)

for idx, demos in df['demographics'].items():
    if len(demos) > 0:  # handles [] naturally
        demo_matrix.loc[idx, demos] = 1

X = demo_matrix.copy()

X = sm.add_constant(X)

y = df['drop_rate']

model = sm.OLS(y, X, missing='drop').fit()

print(model.summary())

results = pd.DataFrame({
    'demographic': model.params.index,
    'coef': model.params.values,
    'pval': model.pvalues.values,
    'ci_low': model.conf_int()[0],
    'ci_high': model.conf_int()[1],
}).query("demographic != 'const'")


if not results.empty:
    results['pval_fdr'] = multipletests(
        results['pval'],
        method='fdr_bh'
    )[1]

    results['significant'] = results['pval_fdr'] < 0.05

else:
    results['pval_fdr'] = pd.Series(dtype=float)
    results['significant'] = pd.Series(dtype=bool)


results = results.sort_values('coef', ascending=False)

results